# Clustering packets of WPT using k-means method - applied to the synthetic data obtained by solving GA
This notebook produces the figures used in the **paper**, using the results of processed synthetic signals. Here we use average enerhy, spectral entropy, kurtosis and log-energy for k-means clustering and thresholding. 

In [ ]:
import sys
if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

In [ ]:
%matplotlib inline
sys.path.append("..")
import os
import pickle

FILE_PATH = os.path.abspath(".")
RELATIVE_PATH = "./paper2024_results/"
folder_path = os.path.abspath(os.path.join(FILE_PATH, RELATIVE_PATH))

import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
from wavelet_transform.utils.save_plots import save_figure

import pywt
# pywt.wavelist(kind='discrete')

import numpy as np
import wavelet_transform.utils.training_data as train
from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.extensions import fft, to_dB, psd, trim_data
from wavelet_transform.utils.machine_learning import train_k_means #,fit_and_evaluate, assess_parameter
import wavelet_transform.utils.kmeans_results as results
from wavelet_transform.utils.preprocessing import preprocess
from wavelet_transform.utils.bubbledynamics import BubbleDynamics as BD

### Parameters
Specify parameters for solving GA and generating synthetic data, and performing wavelet transform. The ultrasound parameters are fixed at 1MHz and 1MPa. Three secarios are considered for the paper, as shown below:
1. transient cavitation with multiple ultrea- and harmonics + BB: r_init = 0.5e-6
2. transient cavitation with BB: r_init = 5e-6
3. stable cavitation: r_init = 50e-6


In [ ]:
# Set the sonication pulse parameters
pac = 1e6  # the amplitude of the sine wave in Pa
frequency = 1.1 * 1e6  # insonation frequency (Hz)
period = 1 / frequency # the period of the acoustic wave at the fundamental frequency 
n_cycles = 100  # no. of cycles of the sine wave
parameters = {
    "time_delay": 0 * period,  # time delay, t1
    "pulse_width": 1000*n_cycles * period,  # pulse width, W = 1 / Q
    "phase_initial": 0 * period,  # tau
    "phase_shift": 0, #np.pi / 4,  # phi
    "no_harmonics": 1,   # k
    "Rdot_init": 0,  # initial bubble's wall velocity in m/s
}

# Set sampling rate and noise std
sampling_rate = frequency * 1e2  # sampling rate to generate the time vector
signal2noise = 30

r_measure = 1e-4 # distance from the centre of bubble which radiated pressure (acoustic emission from bubble) is calculated over time

The R0 (initial bubble radius) values are defined below. We consider the sonication parameters to be fixed and different stable and transient cavitation scenarios are designed by only changing the initial bubble radius.  

In [ ]:
# Scenario 1: transient with multiple ultra- and harmonics + broadband in acoustic emission
Rinit1 = 0.5e-6  # initial bubble radius in meters
# Scenario 2: transient with a harmonic + broadband in acoustic emission
Rinit2 = 3.4e-6  # initial bubble radius in meters
# # Scenario 3: stable with a harmonic in acoustic emission
# Rinit3 = 15e-6  # initial bubble radius in meters
# Scenario 4: stable with fundamental only in acoustic emission
Rinit4 = 50e-6  # initial bubble radius in meters
Rinit_list = [Rinit1, Rinit2, Rinit4]
n_cases = len(Rinit_list)

Rinit_dict = {}
for r_init in Rinit_list:
    Rinit_dict['R0_'+str(r_init*1e6)+'um'] = r_init

result_id = "_".join(Rinit_dict.keys())+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
print(result_id)

results_df = [{"t": [],
               "data": [],
               "data_preprocessed":[],
               "t_data_truncated":[],
               "data_truncated":[],
               "wpt_df":[], 
               "wpt_df_train":[],
               "tr_obj":[],
               "spectrum":[],
               "spectrum_freq":[],
               "kmeans":[],
               "labels":[],
               "labels_sorted":[],
               "centers":[],
               "data_train":[],
               "data_train_normalised":[], 
               "centers":[], 
               "centers_scaled":[],
               "centers_sorted":[],
               "Rinit":list(Rinit_dict.values())[i],
               "legend": list(Rinit_dict.keys())[i]}
              for i in range(n_cases) ]

Setting the wavelet transform parameters and saving/export flag:

In [ ]:
# Parameters for the wavelet transform
wavelet = ["db30", "db30", "db30"]
levels = 8  # set to None to use max-level transform
mode = "symmetric"
use_min_entropy = True
n = 7
min_level = [n, n, n]
freq_threshold = 1 # 10e6 / (sampling_rate / 2)  # the nodes with norm_freq_range above this threshold will not be kept
fft_lim = 10 # this is the upper limit for plotting the spectrum (fft/psd of emissions)

keep_ratio = "valid"

threshold = 3

In [ ]:
# k-means parameters
kmeans_params = {"init": "k-means++",
    "n_init":20,
    "max_iter":1000,
    "tol":1e-20,
    "verbose":0,
    "algorithm":"lloyd",
    }

# training data
classifying_metrics = [
    "log average energy",
    "kurtosis",
    "spectral entropy",
    "average energy",
    # "level entropy",
    # "skewness"
    # "entropy",
    # "crest factor",
]
# Set to None to select optimal number of clusters by silhouette score
use_inverse_metrics = True
inv_keep_ratio = 0.9  # keep ratio for inverse defined only for float input
scale_data = "standard"  # 'standard' to standardise or 'min' to scale by minimum value 


Setting the export flags - for figs and results:

In [ ]:
savefig_flag = True
save_results_flag = True

Producing and plotting the wavelet and scaling functions for the selected wavelet family:

In [ ]:
Font_Size = 24
Tick_Size = 24
Fig_Height = 400
Fig_Width = 800

### Generating acoustic emission data by solving Gilmore-Akulichev model
Generate the acoustic emission data and add measurement noise (assuming additive white Gaussian noise). The parameters are defined above. 

In [ ]:
# BUBBLE DYNAMICS DATA
data_t_tuples = []
data_t_trimmed_tuples = []
data_preprocessed_list = []
i = 0
for Rinit in Rinit_list:
    data_tmp, t_tmp = train.get_data(
        Rinit, frequency, pac, sampling_rate, n_cycles=n_cycles, r_measure=r_measure,
        return_time=True, **parameters
    )
    # Add noise
    data_tmp = train.add_noise(data_tmp, signal2noise, display_std_noise=True)
    # Normalise
    data_preprocessed = preprocess(data_tmp, window=False)
    data_preprocessed_list.append(data_preprocessed)
    # Trimming data for visualisation
    data_trunc, t_data_trunc = trim_data(data_preprocessed, t_tmp, (0*period, 7*period), return_time=True)
    data_t_trimmed_tuples.append([data_trunc, t_data_trunc])
    data_t_tuples.append([data_tmp,t_tmp])
    psd_amp, psd_freq = psd(data_preprocessed, sampling_rate, frequency)
    results_df[i]['data_truncated']= data_trunc
    results_df[i]['t_data_truncated'] = t_data_trunc
    results_df[i]['data_preprocessed'] = data_preprocessed
    results_df[i]['t'] = t_tmp
    results_df[i]['data'] = data_tmp
    results_df[i]['spectrum_freq'] = psd_freq
    results_df[i]['spectrum'] = psd_amp
    i+=1
    

Plotting the acoustic emissions and their spectra. The truncated signals and their spectra are used in the paper.

In [ ]:
data_t_df = pd.concat( [pd.DataFrame(
    data={
        "t": data_t_tuples[i][1]/period,
        "data": data_t_tuples[i][0],
        "legend": list(Rinit_dict.keys())[i],
    }
) for i in range(n_cases) ])

data_t_trimmed_df = pd.concat( [pd.DataFrame(
    data={
        "t": data_t_trimmed_tuples[i][1]/period,
        "data": data_t_trimmed_tuples[i][0],
        "legend": list(Rinit_dict.keys())[i],
    }
) for i in range(n_cases) ])


In [ ]:
fig = px.line(data_t_df,x="t", y="data", color="legend")
fig.update_layout(
    height=Fig_Height,
    width=Fig_Width,
    xaxis=dict(title="t/period [DL]"),
    yaxis=dict(title="Radiated pressure /p<sub>0</sub> [DL]"),
)
fig.show()

In [ ]:
from itertools import cycle, islice
lines = ['dot', 'dashdot', 'solid','dot', 'dash',  'longdash','longdashdot']
colours = ['blue','red','black','cyan','green']
linestyles = cycle(lines)
colournames = cycle(colours)
fig_ae = go.Figure()
# for i in range(2):
#     fig.add_trace( 
#         go.Scatter(x=data_t_trimmed_tuples[i][1]/period, y=data_t_tuples[i][0], line=dict(dash=dash_ops[i]), name=Rinit_list[i])
#     )

for i in range(n_cases):
    fig_ae.add_trace( 
        go.Scatter(x=results_df[i]['t_data_truncated']/period, y=results_df[i]['data_truncated'], 
                   line=dict(dash=next(linestyles),color=next(colournames)), name=results_df[i]['legend'])
    )
    
fig_ae.update_layout(
    height=Fig_Height,
    width=Fig_Width,
    font_size=Font_Size,
    xaxis=dict(title="t/T [DL]",tickfont = dict(size=Tick_Size)),
    yaxis=dict(title="P<sub>rad</sub> / P<sub>0</sub> [DL]",tickfont = dict(size=Tick_Size)),
    plot_bgcolor='white',
    legend=dict(
        x=0.0,
        y=1,
        traceorder='normal',
        font=dict(size=Font_Size,),
    ),
    legend_title=None,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0  #top margin
    )
)
fig_ae.update_xaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)
fig_ae.update_yaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)
fig_ae.show()

if savefig_flag:
    print("figure name: " + "GA_calculated_emissions_"+result_id)
    save_figure(fig_ae, "GA_calculated_emissions_"+result_id, folder_path)
    

In [ ]:
# Plot generated data
# fft_data, freq = fft(data_processed[0], sampling_rate, frequency)
# psd_data_tuples = [ psd(data_processed_list[i], sampling_rate, frequency) for i in range(2)]

# fig = make_subplots(rows=n_cases, cols=1)

# fig.add_trace(
#     go.Scatter(x=psd_freq, y=to_dB(psd_data), 
#                mode="lines", name="fft - amp"),
#     row=1,
#     col=1,
# )
# fig.add_trace(
#     go.Scatter(
#         x=freq, y=np.angle(fft_data, deg=True), mode="lines", name="fft - phase"
#     ),
#     row=2,
#     col=1,
# )

# fig.update_layout(
#     height=600,
#     width=1000,
#     xaxis1=dict(title="Time [s]"),
#     xaxis2=dict(title="Freq [f/f_0]", range=[0, fft_lim]),
#     # xaxis3=dict(title="Freq [f/f_0]", range=[0, fft_lim]),
#     yaxis1=dict(title="Amplitude [DL]"),
#     yaxis2=dict(title="Amplitude [dB]",range=[-30,40]),
#     # yaxis3=dict(title="Amplitude [rad]"),
# )

# fig.update_layout(
#     height=300,
#     width=1000,
#     xaxis=dict(title="Freq [f/f_0]", range=[0, fft_lim]),
#     yaxis=dict(title="Amplitude [dB]",range=[-30,40]),
# )

fig_psd = go.Figure()
linestyles = cycle(lines)
colournames = cycle(colours)

for i in range(n_cases):
    fig_psd.add_trace( 
        go.Scatter(x=results_df[i]['spectrum_freq'], y=10*np.log10(results_df[i]['spectrum']), 
                   line=dict(dash=next(linestyles),color=next(colournames)), name=results_df[i]['legend'])
    )
        
fig_psd.update_layout(
    height=Fig_Height,
    width=Fig_Width,
    font_size=Font_Size,
    xaxis=dict(title="f / f<sub>0</sub> [DL]",tickfont = dict(size=Tick_Size), range=[0, fft_lim]),
    yaxis=dict(title="Amplitude [dB]",tickfont = dict(size=Tick_Size),range=[-40,50]),
    plot_bgcolor='white',
    legend=dict(
        x=0.7,
        y=1,
        traceorder='normal',
        font=dict(size=Font_Size,),
    ),
    legend_title=None,
    margin=go.layout.Margin(
        l=0, #left margin
        r=0, #right margin
        b=0, #bottom margin
        t=0  #top margin
    )
)
fig_psd.update_xaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)
fig_psd.update_yaxes(
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)

fig_psd.show()
if savefig_flag:
    print("figure name: " + "spectrum_GA_calculated_emissions_"+result_id)
    save_figure(fig_psd, "spectrum_GA_calculated_emissions_"+result_id, folder_path)

## Wavelet transform of the preprocessed signal


In [ ]:
for i in range(n_cases):
    results_df[i]['wpt_df'], results_df[i]['tr_obj'] = train.generate_training_data(
        results_df[i]['data_preprocessed'], levels, wavelet[i], mode, 
        include_tr_obj=True, min_entropy=use_min_entropy,
        min_level=min_level[i], freq_threshold=freq_threshold,
        keep_ratio=keep_ratio
    )

### Heisenberg visualisation
Making the Heisenberg plot of the dwpt transform

In [ ]:
# tr_obj.calc_frequencies(sampling_rate)
# fig = plot.heisenberg(tr_obj, max(t_plot), padding="interpolate", use_log_scale=False)

for i in range(n_cases):
    tr_obj = results_df[i]['tr_obj']
    tr_obj.calc_frequencies(sampling_rate)
    fig_hbg = plot.heisenberg(
        tr_obj, 1/sampling_rate*len(tr_obj.data), max_freq=fft_lim*0.0205, 
        padding="repeat", use_log_scale=True, title=results_df[i]['legend'], vmin = -2, vmax = 1, color_map='viridis'
    )
    if savefig_flag is True:
        fig_hbg_name="Heisenberg_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_hbg_name)
        save_figure(fig_hbg, fig_hbg_name, folder_path)

In [ ]:
# nodes = tr_obj.get_leaf_nodes(order="freq")
# nodes_data = np.zeros((len(nodes), len(nodes[0].data)))
# for i, j in enumerate(range(len(nodes) - 1, -1, -1)):  # maybe reverse
#     node = nodes[j]
#     nodes_data[i, :] = np.abs(node.data)  # plot absolute values of wavelet coefficients
# nodes_data.shape

In [ ]:
# from scipy import signal
# t_plot_ds = signal.decimate(t_plot, 16)[:nodes_data.shape[1]]


In [ ]:
# import plotly.graph_objects as go
# fig = go.Figure()
# fig.add_trace(go.Heatmap(z=20*np.log10(nodes_data),
#                          x=np.linspace(0,t_plot[-1]*1e6,nodes_data.shape[1]) ,
#                          y=np.linspace(0,sampling_rate/2/1e6,nodes_data.shape[0]) ,
#                          colorbar={"title": "Amplitude [dB]"},
#                          colorscale='gray'))

# fig.update_layout(
#     autosize=False,
#     width=1000,
#     height=800,
#     xaxis=dict(
#         title_text="time [us]",
#         titlefont=dict(size=20),
#     ),
#     yaxis=dict(
#         # range=[0,sampling_rate/2/1e6],
#         # autorange=False,
#         title_text="frequency [MHz]",
#         titlefont=dict(size=20),
#     )
# )

# # fig.update_yaxes(automargin=True)
# # fig.update_yaxes(autorange="reversed")
# fig.show()

### K-means

k-means clustering on the calculated statistics from the output of the DWPT. The classification metrics used in the paper are: 
    "average energy",
    "kurtosis",
    "log average energy",
    "spectral entropy".

### K-means without PCA

In [ ]:
def clustering(results_df, n_cases, no_clusters, classifying_metrics, kmeans_params,scale_data, metric_sorting=classifying_metrics[0]):
    from wavelet_transform.postprocess.plotting import relabel
    for i in range(n_cases):
        wpt_df = results_df[i]['wpt_df']  # metrics caluclated using the wavelet coefficients
        tr_obj = results_df[i]['tr_obj']
        if use_inverse_metrics is True:
            inv_tr_obj = inv.InverseTransform(tr_obj)  # Use info from inverse transform object instead
            wpt_df_train = inv_tr_obj.info(leaf_nodes=True, freq_threshold=freq_threshold, keep_ratio=inv_keep_ratio)
        else:
            wpt_df_train = wpt_df.copy()
        
        data_train, labels, data_train_normalised, centers, scaler, kmeans = train_k_means(
            wpt_df_train, 
            classifying_metrics, 
            no_clusters=no_clusters, 
            kmeans_params=kmeans_params, 
            scale_data=scale_data
        )
        centers_scaled = scaler.fit_transform(centers)
        centers_sorted, labels_sorted = relabel(labels, centers, metric_sorting, ascending=False)
        results_df[i]['labels']=labels
        results_df[i]['centers']=centers
        results_df[i]['labels_sorted']=labels_sorted
        results_df[i]['centers_sorted']=centers_sorted
        results_df[i]['kmeans']=kmeans
        results_df[i]['centers_scaled']=centers_scaled
        results_df[i]['data_train_normalised']=data_train_normalised
        results_df[i]['data_train']=data_train
        results_df[i]['wpt_df_train']=wpt_df_train

In [ ]:
no_clusters = 2
clustering(results_df, n_cases, no_clusters, classifying_metrics, kmeans_params,scale_data, metric_sorting="average energy")

In [ ]:
from IPython.display import display
display(results_df[0]['centers'])
display(results_df[0]['centers_sorted'])

### Evaluate k-means using a range of metrics 

see scikit learn documentation https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html

In [ ]:
# fit_and_evaluate(kmeans, data_train_normalised, labels, name='wpt data', n_runs=100)

Numerical sensitivity analysis

In [ ]:
# parameter_of_interest = "max_iter"
# values = [i for i in range(1, 100, 2)]
# rand_indices = assess_parameter(
#     parameter_of_interest, values, wpt_df_train, 
#     classifying_metrics, no_clusters
# )

# plt.plot(values, rand_indices)
# plt.title(f"Effect of {parameter_of_interest} on clustering algorithm")
# plt.ylabel("Adjusted rand index")
# plt.xlabel(f"{parameter_of_interest.capitalize()}")
# plt.show()

## Save results

the results_df dicts are saved.

In [ ]:
# Gather db parameters for storage
bd_params = {
    "frequency": frequency,
    "freq_threshold": freq_threshold,
    "fft_lim": fft_lim,
    "pac": pac,
}

# bd_params.update(parameters)
# additional_params = {
#     "raw_data": data,
#     "bd_params": bd_params
# }

if save_results_flag:
    for i in range(n_cases):
        file_name="results_GA_calculated_emissions_enforced2clusters_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        file_name = file_name.replace(".","p")
        file_path = os.path.join(folder_path, file_name + ".pkl")
        with open(file_path, "wb") as file:
            results_df[i].update(bd_params)
            pickle.dump(results_df[i], file)



In [ ]:
# from wavelet_transform.postprocess.plotting import relabel
# ordered_centers, ordered_labels = relabel(results_df[0]['labels'], results_df[0]['centers'], 'average energy', ascending=False)

# Plotting

### 3d plot

3d plots of the clustering results with axes: energy, kurtosis, crest factor

In [ ]:
labels1 = dict(((0,"transient"),(1,"periodic")))
labels2 = dict(((1,"transient"),(0,"periodic")))
label_map = [labels2]*n_cases
# label_map[0]=labels2

In [ ]:
print(label_map)

In [ ]:
for i in range(n_cases):
    classifying_metrics = results_df[i]['data_train'].columns.to_list()
    # ordered_centers, ordered_labels = relabel(results_df[i]['labels'], results_df[i]['centers'], 'average energy', ascending=False)
    fig_cls = plot.plot_clustering(
        results_df[i]['data_train'],
        results_df[i]['centers_sorted'], 
        results_df[i]['labels_sorted'],
        metrics_to_plot=classifying_metrics,
        levels=results_df[i]['wpt_df_train']["level"].tolist(),
        label_map = label_map[i],
        title=results_df[i]['legend']
    )
    if savefig_flag:
        fig_cls_name="clusters_enforced2clusters_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_cls_name)
        save_figure(fig_cls, fig_cls_name, folder_path)
    

In [ ]:
# Plot centroid metrics
for i in range(n_cases):
    fig_metrics = plot.plot_centers(results_df[i]['data_train'].columns.to_list(), 
                        results_df[i]['centers_sorted'],label_map=label_map[i])
    if savefig_flag:
        fig_metrics_name="metrics_centeroids_enforced2clusters_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_metrics_name)
        save_figure(fig_metrics, fig_metrics_name, folder_path)

In [ ]:
for i in range(n_cases):
    fig_psd_rec = plot.plot_fft_by_group(
        results_df[i]['tr_obj'], results_df[i]['labels_sorted'], sampling_rate=sampling_rate,
        clusters=None, 
        xlims=[0,fft_lim],
        ylims=[-100,50],
        fundamental_freq=frequency,
        freq_threshold=freq_threshold, label_map=label_map[i],
        fig_height=Fig_Height,
        title = "FFT comparison - " + results_df[i]['legend']
    )
    if savefig_flag:
        fig_psd_rec_name = "spectrum_rec_centeroids_enforced2clusters_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_psd_rec_name)
        save_figure(fig_psd_rec, fig_psd_rec_name, folder_path)
    
    fig_rec = plot.plot_reconstructions_by_group(
        results_df[i]['tr_obj'], results_df[i]['labels_sorted'], t_plot=results_df[0]['t_data_truncated'], freq_threshold=freq_threshold, 
        label_map=label_map[i], fig_height=Fig_Height, title = "Reconstructed nodes by group - " + results_df[i]['legend']
    )
    if savefig_flag:
        fig_rec_name = "sig_rec_centeroids_enforced2clusters_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_rec_name)
        save_figure(fig_rec, fig_rec_name, folder_path)

Now, to further decompose the transient component, we let the Kmeans algorithm identify the numbers of cluisters based on the classifiers.

In [ ]:
no_clusters = (3, 6)  # upeer bound non-inclusive e.g. 3,6 chooses between 3, 4, 5 clusters
clustering(results_df, n_cases, no_clusters, classifying_metrics, kmeans_params,scale_data,metric_sorting="average energy")


In [ ]:
display(results_df[0]['centers'])
display(results_df[0]['centers_sorted'])

In [ ]:
label_map_details = []         
label_map_details_bar = []  
for i in range(n_cases):
    label_map_details.append(dict((j, f"Cluster {j}") for j in range(len(results_df[i]['centers_sorted']))))
    label_map_details_bar.append(dict((j, f"{j}") for j in range(len(results_df[i]['centers_sorted']))))
print(label_map_details)

In [ ]:
for i in range(n_cases):
    classifying_metrics = results_df[i]['data_train'].columns.to_list()
    fig_cls_arb = plot.plot_clustering(
        results_df[i]['data_train'],
        results_df[i]['centers_sorted'], 
        results_df[i]['labels_sorted'],
        metrics_to_plot=classifying_metrics,
        levels=results_df[i]['wpt_df_train']["level"].tolist(),
        label_map = label_map_details[i],
        title=results_df[i]['legend']
    )
    if savefig_flag:
        fig_cls_arb_name = "clusters_arbitrary_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_cls_arb_name)
        save_figure(fig_cls_arb, fig_cls_arb_name, folder_path)
    

In [ ]:
results_df[0]['centers_sorted'].shape[0]

In [ ]:
# Plot centroid metrics
for i in range(n_cases):
    fig_metric_arb = plot.plot_centers(results_df[i]['data_train'].columns.to_list(), 
                            results_df[i]['centers_sorted'],label_map=label_map_details_bar[i], xlabel_user = "Cluster"
                           )
    print(results_df[i]["centers_sorted"]["kurtosis"])
    if savefig_flag:
        fig_metric_arb_name="metrics_centroids_arbitrary_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_metric_arb_name)
        save_figure(fig_metric_arb, fig_metric_arb_name, folder_path)

### Results

Plotting the reconstructed waves at each node alongside their assigned group number (non-pca).

In [ ]:
# tr_obj.calc_frequencies(sampling_rate)
if tr_obj.max_level() <= 7:
    for i in range(n_cases):
        results_df[i]['tr_obj'].plot_tree(
            node_size="average energy", 
            node_colour=list(results_df[i]['labels_sorted']), 
            freq_threshold=freq_threshold, 
            graph_legend="Graph legend - nodes: average energy",
            fig_name=f"dwpt_graph_{results_df[i]['legend']}",
            sampling_rate=sampling_rate,  # calculate actual frequencys instead of normalised frequencies
            show_frequency=False  # whether to plot the frequency ranges of each node
    )


In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
# fig = plot.plot_fft_by_group(
#     tr_obj, labels, sampling_rate,
#     clusters=None, xlim=10, fundamental_freq=frequency,
#     freq_threshold=freq_threshold
# )

for i in range(n_cases):
    fig_psd_rec_arb = plot.plot_fft_by_group(
        results_df[i]['tr_obj'], results_df[i]['labels_sorted'], sampling_rate=sampling_rate,
        clusters=None, 
        xlims=[0,fft_lim], 
        ylims=[-100,50],
        fundamental_freq=frequency,
        freq_threshold=freq_threshold, label_map=label_map_details[i],
        fig_height=Fig_Height,
        title = "FFT comparison - " + results_df[i]['legend']
    )
    if savefig_flag:
        fig_psd_rec_arb_name = "spectrum_rec_centeroids_arbitrary_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_psd_rec_arb_name)
        save_figure(fig_psd_rec_arb, fig_psd_rec_arb_name, folder_path)
    
    fig_rec_arb = plot.plot_reconstructions_by_group(
        results_df[i]['tr_obj'], results_df[i]['labels_sorted'], t_plot=results_df[0]['t_data_truncated'], freq_threshold=freq_threshold, 
        fig_height=Fig_Height, title = "Reconstructed nodes by group - " + results_df[i]['legend']#, label_map=label_map_details[i] 
    )
    if savefig_flag:
        fig_rec_arb_name = "sig_rec_centeroids_arbitrary_GA_calculated_emissions_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        print("figure name: " + fig_rec_arb_name)
        save_figure(fig_rec_arb, fig_rec_arb_name, folder_path)

pca results

In [ ]:
if save_results_flag:
    for i in range(n_cases):
        file_name="results_GA_calculated_emissions_arbitrary clusters_"+results_df[i]['legend']+f"_freq_{frequency*1e-6:.1f}MHz_Pac_{pac*1e-6:.1f}MPa"
        file_name = file_name.replace(".","p")
        file_path = os.path.join(folder_path, file_name + ".pkl")
        with open(file_path, "wb") as file:
            # results_df[i].update(bd_params)
            pickle.dump(results_df[i], file)

separately plot reconstructions of all nodes belonging to selected clusters

In [ ]:
# import wavelet_transform.wpt.inverse_transform as inv
# # Add number to list to plot reconstructions of all nodes belonging to the cluster

# for i in range(n_cases):
#     plot_centres = []
#     for i, (node, assignment) in enumerate(zip(results_df[i]['tr_obj'].get_leaf_nodes(freq_threshold=freq_threshold), 
#                                                results_df[i]['labels'])):
#         if assignment in plot_centres:
#             print("spectral entropy:", node.spectral_entropy)
#             print("crest factor", node.crest_factor)
#             rec = inv.inv_dwpt(results_df[i]['tr_obj'], node.level, nodes=node)
#             fft_rec, freq = fft(rec, sampling_rate, frequency)
#             plt.plot(freq/frequency, np.log10(np.abs(fft_rec)))
#             plt.xlim(0, 5)
#             plt.title(f"frequency range: {node.frequency_range}, fft")
#             plt.show()
#             plt.plot(rec)
#             plt.title(f"reconstructed signal")
#             plt.show()


In [ ]:
import wavelet_transform.utils.statistical_features as sf


i=1
rec_high_kurt, rec_low_kurt = plot.get_reconstructions_by_threshold(
    results_df[i]['tr_obj'], feature="kurtosis", threshold=threshold, freq_threshold=freq_threshold
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    # x=freq, 
    y=results_df[i]['tr_obj'].data,
    mode='lines',
    name="original signal"
))
fig.add_trace(go.Scatter(
    # x=freq, 
    y=rec_high_kurt,
    mode='lines',
    name="high kurtosis reconstruction (wavelet coefficient kurtosis)"
))
fig.add_trace(go.Scatter(
    # x=freq, 
    y=rec_low_kurt,
    mode='lines',
    name="low kurtosis reconstruction (wavelet coefficient kurtosis)"
))
fig.show()

rec_high_kurt2, rec_low_kurt2 = plot.get_reconstructions_by_threshold(
    results_df[i]['tr_obj'], feature=sf.kurtosis, threshold=threshold
)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    # x=freq, 
    y=results_df[i]['tr_obj'].data,
    mode='lines',
    name="original signal"
))
fig2.add_trace(go.Scatter(
    # x=freq, 
    y=rec_high_kurt2,
    mode='lines',
    name="high kurtosis reconstruction (reconstruction kurtosis)"
))
fig2.add_trace(go.Scatter(
    # x=freq, 
    y=rec_low_kurt2,
    mode='lines',
    name="low kurtosis reconstruction (reconstruction kurtosis)"
))
fig2.show()

In [ ]:
import wavelet_transform.wpt.inverse_transform as inv
import matplotlib.pyplot as plt

plot_centres = np.arange(len(results_df[i]['centers_sorted'])) 
plot_centres = plot_centres[results_df[i]['centers_sorted']['kurtosis']>threshold]
# Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
# plot_centres = [1]


reconstructions = [np.zeros(len(results_df[i]['tr_obj'].data)) for _ in range(2)]

# plt.figure()
for ii, (node, assignment) in enumerate(zip(results_df[i]['tr_obj'].get_leaf_nodes(freq_threshold=freq_threshold), 
                                            results_df[i]['labels_sorted'])):
    if assignment in plot_centres:
        # List reconstructions
        rec_signal = inv.inv_dwpt(results_df[i]['tr_obj'], node.level, nodes=node)
        sig_kurt = sf.kurtosis(rec_signal)
        j = 0 if sig_kurt > threshold else 1  # shorten name for readability 
        print("kurtosis:", sig_kurt, "\n assignment:",assignment," \n j:",j,"ii",ii)               
        reconstructions[j] = reconstructions[j] + rec_signal
        #TODO take a look at the specific kurtosis of the separate reconstructions and then the kurtosis of their sum
        # psd_rec, freq = psd(rec, sampling_rate, frequency)
        # freq /= frequency
        # plt.plot(frequency, to_dB(psd_rec))
        # plt.xlim(0, fft_lim)
        # plt.title(f"frequency range: {np.divide(node.frequency_range,frequency)} (normalised by f0)")
        # plt.show()
        # plt.plot(reconstructions[0])
        # plt.plot(reconstructions[1])
        # plt.show()

# fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=reconstructions[0],
        mode="lines",
        name="higher than threshold-clusters",
    )
)
fig.add_trace(
    go.Scatter(
        y=reconstructions[1],
        mode="lines",
        name="lower than threshold-clusters",
    )
)


In [ ]:
results_df[i]['centers_sorted']

In [ ]:
psd_reconstructions_higher, freq_higher = psd(reconstructions[0], sampling_rate, frequency)
psd_reconstructions_lower, freq_lower = psd(reconstructions[1], sampling_rate, frequency)
psd_rec_all_kurt_thrsh_higher, freq2_higher = psd(rec_high_kurt2, sampling_rate, frequency)
psd_rec_all_kurt_thrsh_lower, freq2_lower = psd(rec_low_kurt2, sampling_rate, frequency)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq2_higher,
        y=to_dB(psd_rec_all_kurt_thrsh_higher),
        mode="lines",
        name="higher than threshold_all nodes",
    )
)
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_reconstructions_higher),
        mode="lines",
        name="higher than threshold_ specified clusters",
    )
)


fig.add_trace(
    go.Scatter(
        x=freq2_lower,
        y=to_dB(psd_rec_all_kurt_thrsh_lower),
        mode="lines",
        name="lower than threshold_all nodes",
    )
)
fig.add_trace(
    go.Scatter(
        x=freq_lower,
        y=to_dB(psd_reconstructions_lower),
        mode="lines",
        name="lower than threshold_ specified clusters",
    )
)

fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
)

In [ ]:
import scipy.ndimage as sp

In [ ]:
print(reconstructions[1])

In [ ]:
psd_curve = sp.gaussian_filter1d(psd_reconstructions_higher, 3)

# fig = go.Figure()
# fig.add_trace(
#     go.Scatter(x=freq_higher, y=to_dB(psd_curve), name="envelope")
# )
# fig.add_trace(
#     go.Scatter(x=freq_higher, y=to_dB(psd_reconstructions_higher), name="signal")
# )
# fig.show()

rms = np.sqrt(np.mean(psd_curve**2))
half_peak = max(psd_curve) / 2
e_peak = max(psd_curve) / np.e**2

broadband_lines = {
    "rms": rms, 
    "half peak": half_peak, 
    "1/e^2 peak": e_peak
}

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_reconstructions_higher),
        mode="lines",
        name="higher than threshold",
    )
)

fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_curve),
        name="envelope"
    )
)

for line_name, line in broadband_lines.items():
    db_line = 10 * np.log10(line)
    fig.add_trace(
        go.Scatter(
            x=freq_higher,
            y=db_line*np.ones(len(freq_higher)),
            line=dict(dash="dash"),
            name=line_name
        )
    )
fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
)


In [ ]:
print("Energy of the HIGH-kurtosis broadband [dB]: ",sf.log_avg_energy(reconstructions[0]), 
      "\nEnergy of the LOW-kurtosis broadband [dB]:",sf.log_avg_energy(reconstructions[1]))

In [ ]:
print("Energy of the HIGH-kurtosis broadband [dB] - all node based calculation: ",sf.log_avg_energy(rec_high_kurt), 
      "\nEnergy of the LOW-kurtosis broadband [dB] - all node based calculation:",sf.log_avg_energy(rec_low_kurt))

In [ ]:
print("Spectral entropy of the HIGH-kurtosis broadband [dB]: ",sf.spectral_entropy(reconstructions[0]), 
      "\nSpectral entropy of the LOW-kurtosis broadband [dB]:",sf.spectral_entropy(reconstructions[1]))

In [ ]:
print("Kurtosis of the HIGH-kurtosis broadband [dB]: ",sf.kurtosis(reconstructions[0]), 
      "\nKurtosis of the LOW-kurtosis broadband [dB]:",sf.kurtosis(reconstructions[1]))